-----

### **Clase Avanzada: El Motor Cinemático - Animación, Cámaras y Video**

**Objetivo:** Extender nuestro motor para convertirlo en una herramienta de animación no interactiva. Se implementará un **secuenciador de eventos (Timeline)** para guionizar escenas y un **grabador de video** para exportar el resultado final a un archivo `.mp4`.

-----

### **1. La Herramienta Externa Esencial: FFmpeg**

Para el paso final de crear el video, necesitaremos una herramienta externa.

  * **¿Qué es?** FFmpeg es el programa de código abierto estándar para procesar video.
  * **¿Cómo se instala?** Descárgalo desde la página oficial **`ffmpeg.org`**. Para que funcione, debes añadir la ubicación de `ffmpeg.exe` a la variable de entorno PATH de tu sistema, o, más fácil, **copiar el archivo `ffmpeg.exe` en la carpeta raíz de tu proyecto de animación**.

-----

### **2. La Estructura de Archivos Modularizada**

Crea una carpeta para este nuevo proyecto (ej: `animator/`) y organiza tus archivos de la siguiente manera:

```
/tu_proyecto/
|
|-- engine/                 # <-- Nuestra carpeta de motor existente
|   |-- game_object.py      # <-- Versión actualizada
|   |-- timeline.py         # <-- ¡NUEVO!
|   |-- recorder.py         # <-- ¡NUEVO!
|   |-- ... (los otros módulos del motor)
|
|-- animator/               # <-- ¡NUEVO PROYECTO!
|   |-- entities/
|   |   |-- __init__.py
|   |   |-- actor.py        # <-- Entidades específicas para la animación
|   |
|   |-- main.py             # <-- El script principal que ejecuta la animación
|
|-- ffmpeg.exe              # <-- Copia aquí el ejecutable de FFmpeg
```

-----

### **3. El Código Completo por Archivos**

#### **Archivo 1: `engine/game_object.py` (Actualizado)**

*Añadimos un vector `pos` para permitir movimientos suaves y decimales, crucial para las animaciones.*

```python
# engine/game_object.py
import pygame

class GameObject(pygame.sprite.Sprite):
    """
    Clase base que maneja imágenes estáticas y animaciones.
    Ahora incluye un vector de posición para movimientos suaves.
    """
    def __init__(self, x, y, frames):
        super().__init__()
        
        if isinstance(frames, list):
            self.frames = frames
            self.is_animated = len(frames) > 1
            self.current_frame = 0
            self.image = self.frames[self.current_frame]
            self.anim_speed = 0.2
            self.anim_timer = 0
        else:
            self.frames = [frames]
            self.is_animated = False
            self.image = frames

        self.rect = self.image.get_rect(center=(x, y))
        # --- CAMBIO CLAVE: Vector de posición para movimientos decimales ---
        self.pos = pygame.math.Vector2(x, y)

    def update(self, *args, **kwargs):
        """El método update ahora también se encarga de la animación."""
        self.update_animation()
        # Sincroniza el rect con el vector de posición
        self.rect.center = self.pos

    def update_animation(self):
        if not self.is_animated:
            return

        self.anim_timer += self.anim_speed
        if self.anim_timer >= 1:
            self.anim_timer = 0
            self.current_frame = (self.current_frame + 1) % len(self.frames)
            old_center = self.rect.center
            self.image = self.frames[self.current_frame]
            self.rect = self.image.get_rect(center=old_center)
```

#### **Archivo 2: `engine/timeline.py` (¡NUEVO\!)**

*Este es el guion de nuestra película.*

```python
# engine/timeline.py
import pygame

class Timeline:
    """Gestiona una secuencia de eventos cinemáticos."""
    def __init__(self, scene):
        self.scene = scene
        self.events = []
        self.current_event_index = -1
        self.active_event = None
        self.is_playing = False

    def add_event(self, event):
        event.scene = self.scene
        self.events.append(event)

    def play(self):
        print("Timeline: Reproducción iniciada.")
        self.is_playing = True
        self.current_event_index = -1
        self.next_event()

    def next_event(self):
        self.current_event_index += 1
        if self.current_event_index < len(self.events):
            self.active_event = self.events[self.current_event_index]
            print(f"Timeline: Iniciando evento {self.current_event_index + 1}/{len(self.events)} ({type(self.active_event).__name__})")
            self.active_event.start()
        else:
            self.active_event = None
            self.is_playing = False
            print("Timeline: Secuencia finalizada.")

    def update(self, dt):
        if not self.is_playing or not self.active_event:
            return
        
        self.active_event.update(dt)
        if self.active_event.is_finished():
            self.next_event()

# --- Clases de Eventos ---
class Event:
    """Clase base para todos los eventos de la timeline."""
    def start(self): self.finished = False
    def update(self, dt): pass
    def is_finished(self): return self.finished

class WaitEvent(Event):
    """Espera una cantidad de tiempo determinada."""
    def __init__(self, duration_seconds):
        self.duration = duration_seconds
        self.timer = 0
    def start(self):
        super().start()
        self.timer = 0
    def update(self, dt):
        self.timer += dt
        if self.timer >= self.duration: self.finished = True

class MoveEvent(Event):
    """Mueve un actor a una posición objetivo durante un tiempo determinado."""
    def __init__(self, character_name, target_pos, duration):
        self.char_name = character_name
        self.target_pos = pygame.math.Vector2(target_pos)
        self.duration = max(0.01, duration) # Evitar división por cero
        self.timer = 0
    def start(self):
        super().start()
        self.character = self.scene.actors[self.char_name]
        self.start_pos = self.character.pos.copy()
    def update(self, dt):
        self.timer += dt
        progress = min(1.0, self.timer / self.duration)
        # Interpolación lineal (lerp) para un movimiento suave
        self.character.pos = self.start_pos.lerp(self.target_pos, progress)
        if progress >= 1.0: self.finished = True

class SetAnimationEvent(Event):
    """Cambia la animación de un actor."""
    def __init__(self, character_name, animation_name):
        self.char_name = character_name
        self.anim_name = animation_name
    def start(self):
        super().start()
        # Asume que el actor tiene un método set_animation
        self.scene.actors[self.char_name].set_animation(self.anim_name)
        self.finished = True # Este evento es instantáneo
```

#### **Archivo 3: `engine/recorder.py` (¡NUEVO\!)**

*Esta es nuestra "cámara de cine" que graba los fotogramas.*

```python
# engine/recorder.py
import pygame
import os
import subprocess

class FrameRecorder:
    def __init__(self, output_folder="_output_frames"):
        self.output_folder = output_folder
        self.frame_count = 0
        
        # Crear carpeta de salida y limpiar frames antiguos
        os.makedirs(self.output_folder, exist_ok=True)
        print(f"Limpiando carpeta de frames: '{self.output_folder}'")
        for f in os.listdir(self.output_folder):
            os.remove(os.path.join(self.output_folder, f))

    def capture(self, surface):
        """Guarda la superficie actual como un frame PNG numerado."""
        filename = os.path.join(self.output_folder, f"frame_{self.frame_count:06d}.png")
        pygame.image.save(surface, filename)
        self.frame_count += 1

    def compile_video(self, output_filename="output.mp4", fps=60):
        """Usa FFmpeg para unir los frames en un archivo de video."""
        if self.frame_count == 0:
            print("No se capturaron frames, no se puede compilar el video.")
            return

        print(f"Compilando {self.frame_count} frames en un video con FFmpeg...")
        input_pattern = os.path.join(self.output_folder, "frame_%06d.png")
        
        # Comando de FFmpeg
        command = [
            'ffmpeg',
            '-r', str(fps),              # Frames por segundo de entrada
            '-i', input_pattern,         # Patrón de archivos de imagen
            '-c:v', 'libx264',           # Codec de video H.264
            '-crf', '20',                # Factor de calidad (18-28 es un buen rango)
            '-pix_fmt', 'yuv420p',       # Formato de píxeles para máxima compatibilidad
            '-y',                        # Sobrescribir archivo de salida si existe
            output_filename
        ]
        
        try:
            # Ocultar la salida de consola de ffmpeg
            subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            print(f"\n¡Video guardado exitosamente como '{output_filename}'!")
        except (subprocess.CalledProcessError, FileNotFoundError):
            print("\n" + "="*60)
            print("ERROR: No se pudo compilar el video con FFmpeg.")
            print("Asegúrate de que FFmpeg esté instalado y en el PATH de tu sistema,")
            print("o que 'ffmpeg.exe' esté en la misma carpeta que tu script.")
            print("="*60 + "\n")
```

#### **Archivo 4: `animator/entities/actor.py`**

*Una entidad simple para nuestro ejemplo.*

```python
# animator/entities/actor.py
import pygame
from engine.game_object import GameObject

class Actor(GameObject):
    def __init__(self, x, y, color=(255,255,255), size=(50,50)):
        image = pygame.Surface(size)
        image.fill(color)
        super().__init__(x, y, image)

    def update(self):
        # La posición es controlada por la Timeline, pero la animación
        # (si la tuviera) se actualizaría aquí.
        super().update()
```

#### **Archivo 5: `animator/main.py` (El Ejemplo Completo)**

*Este es el script principal que ejecutas. Define la escena y el guion.*

```python
# animator/main.py
import pygame
from engine.timeline import Timeline, WaitEvent, MoveEvent
from engine.recorder import FrameRecorder
from entities.actor import Actor

# --- Configuración de la Animación ---
WIDTH, HEIGHT = 1280, 720
FPS = 60

class AnimationScene:
    """Define los actores y la secuencia de eventos de nuestra animación."""
    def __init__(self):
        # 1. Creamos a nuestros "actores" y los guardamos en un diccionario por nombre
        self.actors = {
            "Obrero_1": Actor(100, 700, color=(200, 50, 50), size=(30, 60)),
            "Viga": Actor(640, 100, color=(150, 150, 150), size=(400, 20))
        }
        
        # 2. Escribimos el "guion" de la animación usando la Timeline
        self.timeline = Timeline(self)
        self.timeline.add_event(WaitEvent(0.5))
        self.timeline.add_event(MoveEvent("Obrero_1", (300, 500), duration=2.0))
        self.timeline.add_event(WaitEvent(0.2))
        self.timeline.add_event(MoveEvent("Obrero_1", (300, 300), duration=1.5))
        self.timeline.add_event(WaitEvent(0.5))
        self.timeline.add_event(MoveEvent("Obrero_1", (980, 300), duration=3.0))
        self.timeline.add_event(WaitEvent(1.0))
        
    def update(self, dt):
        """Actualiza la timeline y los actores en cada fotograma."""
        self.timeline.update(dt)
        for actor in self.actors.values():
            actor.update()

    def draw(self, surface):
        """Dibuja la escena en cada fotograma."""
        surface.fill((20, 20, 30)) # Fondo oscuro
        # Simulamos un edificio en construcción
        pygame.draw.rect(surface, (100, 100, 100), (0, 0, 50, HEIGHT))
        pygame.draw.rect(surface, (100, 100, 100), (WIDTH - 50, 0, 50, HEIGHT))
        
        for actor in self.actors.values():
            surface.blit(actor.image, actor.rect)

# --- Bucle Principal de Renderizado ---
def main():
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    clock = pygame.time.Clock()
    
    scene = AnimationScene()
    recorder = FrameRecorder()
    
    # Iniciamos la animación
    scene.timeline.play()
    
    # El bucle se ejecuta solo mientras la animación esté activa
    while scene.timeline.is_playing:
        dt = clock.tick(FPS) / 1000.0 # Delta time en segundos
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                scene.timeline.is_playing = False
        
        scene.update(dt)
        scene.draw(screen)
        pygame.display.flip()
        
        # Capturamos el fotograma después de que se ha mostrado en pantalla
        recorder.capture(screen)
        
    # Al finalizar, compilamos el video y cerramos
    recorder.compile_video(fps=FPS, output_filename="mi_animacion.mp4")
    pygame.quit()

if __name__ == '__main__':
    main()
```